# Colab Rerank + Evaluate

Use this notebook after candidate generation/training is already done.

It reruns only:
- `scripts/02_rerank.py`
- `scripts/03_evaluate.py`

This is the notebook to run in VSCode with the Colab extension after you log in manually and select the Python kernel.

Default behavior:
- reuses existing `*_scores.parquet`
- uses the shared Drive workspace under `carbon-aware-recsys-colab`
- uses the denser `0.90` to `1.00` lambda tail from the repo config
- reruns all three models across all categories

If you want a single category only, set `CATEGORY = 'electronics'` (or another category) in the config cell.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

DRIVE_BASE = Path('/content/drive/MyDrive')
REPO_URL = 'https://github.com/andersvestrum/carbon-aware-recsys.git'
REPO_BRANCH = 'big_man_thing'
REPO_ROOT = DRIVE_BASE / 'carbon-aware-recsys'
DRIVE_ROOT = DRIVE_BASE / 'carbon-aware-recsys-colab'

RESULTS_DIR = DRIVE_ROOT / 'results'
FIGURES_DIR = DRIVE_ROOT / 'figures'
INTERIM_DIR = DRIVE_ROOT / 'data' / 'interim'

MODELS = ['BPR', 'NeuMF', 'LightGCN']
CATEGORY = None  # None = all categories; or set 'electronics', 'home_and_kitchen', 'sports_and_outdoors'

def clone_repo():
    if REPO_ROOT.exists():
        print(f'Removing existing directory: {REPO_ROOT}')
        shutil.rmtree(REPO_ROOT)
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)],
        check=True,
    )

if (REPO_ROOT / '.git').exists():
    try:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
    except subprocess.CalledProcessError:
        print('Existing checkout could not switch cleanly; re-cloning the repo.')
        clone_repo()
else:
    clone_repo()

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')], check=True)
subprocess.run(
    [
        sys.executable,
        '-c',
        'import recbole, kmeans_pytorch, torch; print(recbole.__version__, torch.__version__)',
    ],
    check=True,
)

print({
    'repo_root': str(REPO_ROOT),
    'results_dir': str(RESULTS_DIR),
    'figures_dir': str(FIGURES_DIR),
    'interim_dir': str(INTERIM_DIR),
    'repo_branch': REPO_BRANCH,
    'models': MODELS,
    'category': CATEGORY,
})


In [ ]:
ALL_CATEGORIES = ['electronics', 'home_and_kitchen', 'sports_and_outdoors']
categories = [CATEGORY] if CATEGORY else ALL_CATEGORIES

missing = []
for category in categories:
    for model in MODELS:
        score_path = RESULTS_DIR / f'{category}_{model}_scores.parquet'
        if not score_path.exists():
            missing.append(str(score_path))

if missing:
    raise FileNotFoundError('Missing score parquet files:\n' + '\n'.join(missing))

print('All required score parquet files are present.')


In [ ]:
import subprocess
import sys

for model in MODELS:
    cmd = [
        sys.executable,
        str(REPO_ROOT / 'scripts' / '02_rerank.py'),
        '--model', model,
        '--results-dir', str(RESULTS_DIR),
        '--interim-dir', str(INTERIM_DIR),
    ]
    if CATEGORY:
        cmd.extend(['--category', CATEGORY])
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)


In [ ]:
import subprocess
import sys

for model in MODELS:
    cmd = [
        sys.executable,
        str(REPO_ROOT / 'scripts' / '03_evaluate.py'),
        '--model', model,
        '--results-dir', str(RESULTS_DIR),
        '--figures-dir', str(FIGURES_DIR),
    ]
    if CATEGORY:
        cmd.extend(['--category', CATEGORY])
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)


In [ ]:
import subprocess

subprocess.run(
    [
        'bash',
        '-lc',
        f"ls -lt '{FIGURES_DIR}' | head -n 40",
    ],
    check=True,
)


In [ ]:
import subprocess

zip_path = DRIVE_ROOT / 'figures_rerank_eval.zip'
subprocess.run(
    [
        'bash',
        '-lc',
        f"cd '{DRIVE_ROOT}' && rm -f '{zip_path.name}' && zip -r '{zip_path.name}' figures",
    ],
    check=True,
)
print({'zip_path': str(zip_path)})
